# 3. Esperimenti e VAE

Usare lo stesso split e la stessa degradazione rende il confronto corretto.


In [1]:
from pathlib import Path
import sys, torch, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import EuroSAT
ROOT=Path.cwd()
while not (ROOT/'README.md').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from eurosat_denoising.utils import *
seed_everything(42); DATA=ROOT/'data'; OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True); DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(ROOT,DEVICE)
dataset=EuroSAT(DATA,download=False); n=len(dataset); sizes=[int(.70*n),int(.15*n),n-int(.85*n)]
a,b,c=random_split(dataset,sizes,generator=torch.Generator().manual_seed(42)); corrupt=lambda x:add_gaussian_noise(x,.15)
train=NoisyCleanDataset(dataset,a.indices,corrupt); val=NoisyCleanDataset(dataset,b.indices,corrupt,1000); test=NoisyCleanDataset(dataset,c.indices,corrupt,2000)
tr=DataLoader(train,128,shuffle=True); va=DataLoader(val,128); te=DataLoader(test,128)


/home/alessandro/Desktop/eurosat-denoising cpu


In [2]:
from torch import nn
class VAE(nn.Module):
 def __init__(self,z=64):
  super().__init__();self.enc=nn.Sequential(nn.Conv2d(3,32,3,2,1),nn.ReLU(),nn.Conv2d(32,64,3,2,1),nn.ReLU());self.mu=nn.Linear(64*16*16,z);self.logvar=nn.Linear(64*16*16,z);self.fc=nn.Linear(z,64*16*16);self.dec=nn.Sequential(nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(),nn.ConvTranspose2d(32,3,4,2,1),nn.Sigmoid())
 def forward(self,x):
  h=self.enc(x).flatten(1);mu,lv=self.mu(h),self.logvar(h);z=mu+torch.exp(.5*lv)*torch.randn_like(mu);return self.dec(self.fc(z).view(-1,64,16,16)),mu,lv
def loss_fn(pred,target,mu,lv,beta=1e-4):
 rec=nn.functional.mse_loss(pred,target);kl=-.5*torch.mean(1+lv-mu**2-lv.exp());return rec+beta*kl,rec,kl
vae=VAE().to(DEVICE);opt=torch.optim.Adam(vae.parameters(),1e-3)
for e in range(1,4):
 vae.train();total=0
 for noisy,clean in tr:
  pred,mu,lv=vae(noisy.to(DEVICE));loss,_,_=loss_fn(pred,clean.to(DEVICE),mu,lv);opt.zero_grad();loss.backward();opt.step();total+=loss.item()
 print('epoch',e,'loss',total/len(tr))


epoch 1 loss 0.010928843695846563
epoch 2 loss 0.006122335225243021
epoch 3 loss 0.0048761131978165865


In [3]:
vae.eval();m=p=0;count=0
with torch.no_grad():
 for noisy,clean in te:
  pred,_,_=vae(noisy.to(DEVICE));pred=pred.cpu();k=len(noisy);count+=k;m+=mse_score(pred,clean)*k;p+=psnr_score(pred,clean)*k
print('VAE test MSE',m/count,'PSNR',p/count,'dB')
print('Confrontare questi valori con il notebook 02 e discutere dettaglio vs regolarizzazione latente.')


VAE test MSE 0.0042429955331263714 PSNR 23.744622736218567 dB
Confrontare questi valori con il notebook 02 e discutere dettaglio vs regolarizzazione latente.
